# Local Features PCA Analysis

Offline analysis of `local_feats` tensors saved by the trainer dump hook.

**Prerequisites:**
1. Set `analysis.dump_local_feats: true` in your config and run training for at least one step.
2. The `.pt` file is written to `output/analysis/local_feats_<subject>.pt`.

**Figures produced (inline):**
- Summary statistics table
- Per-channel mean ± std bar chart
- Channel L2-norm distribution histogram
- Scree plot (per-PC and cumulative explained variance)
- PC1 vs PC2 scatter (coloured by Gaussian index)
- PC1–PC3 3-D scatter
- Pairwise scatter matrix (first 4 PCs)
- PC loadings heatmap (top-N channels per PC)

## 1 · Configuration
Edit the two variables below to point to your `.pt` file.

In [ ]:
from pathlib import Path

# ── Edit these ────────────────────────────────────────────────────────────────
FEAT_PATH   = Path("../output/analysis/local_feats_0001.pt")  # path to .pt file
N_COMPONENTS = 50   # number of principal components to compute
# ──────────────────────────────────────────────────────────────────────────────

# Auto-discover if the explicit path doesn't exist
if not FEAT_PATH.exists():
    candidates = sorted(Path("../output/analysis").glob("local_feats_*.pt"))
    if not candidates:
        raise FileNotFoundError(
            "No local_feats_*.pt files found. "
            "Enable analysis.dump_local_feats in your config and run training first."
        )
    FEAT_PATH = candidates[0]
    print(f"Auto-selected: {FEAT_PATH}")

SUBJECT = FEAT_PATH.stem.replace("local_feats_", "")
print(f"Subject : {SUBJECT}")
print(f"File    : {FEAT_PATH.resolve()}")

## 2 · Imports & helpers

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 – registers 3-D projection

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})


def load_feat(path: Path) -> torch.Tensor:
    """Load a trainer-dumped local_feats tensor → float32 (N, C)."""
    t = torch.load(path, map_location="cpu", weights_only=True)
    if t.ndim == 3:
        t = t[0]          # drop singleton batch dim: (1, N, C) → (N, C)
    if t.ndim != 2:
        raise ValueError(f"Expected 2-D tensor, got shape {tuple(t.shape)}")
    return t.float()


def run_pca(X: torch.Tensor, n_components: int):
    """PCA via torch.pca_lowrank. Returns (scores, ev_ratio, components).

    scores      : (N, n_components)  – coordinates in PC space
    ev_ratio    : (n_components,)    – fraction of total variance per PC
    components  : (n_components, C)  – PC loading vectors (rows)
    """
    n_components = min(n_components, X.shape[0], X.shape[1])
    X_c = X - X.mean(dim=0, keepdim=True)
    U, S, V = torch.pca_lowrank(X_c, q=n_components, niter=4, center=False)
    scores     = (U * S.unsqueeze(0)).numpy()           # (N, k)
    variance   = (S ** 2) / (X.shape[0] - 1)
    total_var  = (X_c ** 2).sum() / (X.shape[0] - 1)
    ev_ratio   = (variance / total_var.clamp_min(1e-12)).numpy()  # (k,)
    components = V.T.numpy()                            # (k, C)
    return scores, ev_ratio, components


print("Imports OK")

## 3 · Load data & summary statistics

In [ ]:
X = load_feat(FEAT_PATH)
X_np = X.numpy()
N, C = X_np.shape

print(f"{'Gaussians (N)':<25} {N}")
print(f"{'Channels  (C)':<25} {C}")
print(f"{'Global mean':<25} {X_np.mean():.5f}")
print(f"{'Global std':<25} {X_np.std():.5f}")
print(f"{'Global min':<25} {X_np.min():.5f}")
print(f"{'Global max':<25} {X_np.max():.5f}")
print(f"{'L2 norm (mean)':<25} {np.linalg.norm(X_np, axis=1).mean():.4f}")

## 4 · Per-channel statistics

In [ ]:
MAX_SHOW = 128
show = min(C, MAX_SHOW)
ch_means = X_np[:, :show].mean(axis=0)
ch_stds  = X_np[:, :show].std(axis=0)
ch_idx   = np.arange(show)

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# ── Mean ± std bar chart ──────────────────────────────────────────────────────
ax = axes[0]
ax.bar(ch_idx, ch_means, yerr=ch_stds, color="teal", alpha=0.7,
       ecolor="gray", capsize=0, width=1.0)
ax.axhline(0, color="black", linewidth=0.6)
ax.set_xlim(-1, show)
ax.set_xlabel(f"Channel index  (showing first {show} of {C})")
ax.set_ylabel("Mean ± std")
ax.set_title(f"Per-channel mean ± std  –  subject {SUBJECT}")

# ── Std-only bar chart (to spot collapsed channels) ──────────────────────────
ax2 = axes[1]
ax2.bar(ch_idx, ch_stds, color="coral", alpha=0.8, width=1.0)
ax2.set_xlim(-1, show)
ax2.set_xlabel(f"Channel index  (showing first {show} of {C})")
ax2.set_ylabel("Std")
ax2.set_title("Per-channel standard deviation  (low = inactive channel)")

fig.suptitle(f"Channel statistics  –  subject {SUBJECT}", fontsize=12, y=1.01)
fig.tight_layout()
plt.show()

## 5 · Feature vector L2-norm distribution

In [ ]:
norms = np.linalg.norm(X_np, axis=1)   # (N,)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(norms, bins=60, color="mediumpurple", alpha=0.8, edgecolor="white", linewidth=0.3)
ax.axvline(norms.mean(), color="red", linewidth=1.5, linestyle="--",
           label=f"mean = {norms.mean():.3f}")
ax.axvline(np.median(norms), color="orange", linewidth=1.5, linestyle="--",
           label=f"median = {np.median(norms):.3f}")
ax.set_xlabel("L2 norm of feature vector")
ax.set_ylabel("Count")
ax.set_title(f"Gaussian feature-vector norm distribution  –  subject {SUBJECT}")
ax.legend()
fig.tight_layout()
plt.show()

## 6 · Run PCA

In [ ]:
scores, ev_ratio, components = run_pca(X, N_COMPONENTS)
k = len(ev_ratio)
cumulative = np.cumsum(ev_ratio)

print(f"Computed {k} principal components\n")
print(f"{'PC':>4}  {'Var (%)':>9}  {'Cumul (%)':>10}")
print("-" * 30)
for i in range(min(k, 20)):
    print(f"{i+1:>4}  {ev_ratio[i]*100:>9.3f}  {cumulative[i]*100:>10.3f}")
if k > 20:
    print(f"  ... top {k} PCs explain {cumulative[-1]*100:.1f}% of total variance")

## 7 · Scree plot

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 4))

xs = np.arange(1, k + 1)
ax1.bar(xs, ev_ratio * 100, color="steelblue", alpha=0.75, width=0.8)
ax1.set_xlabel("Principal Component")
ax1.set_ylabel("Explained Variance (%)", color="steelblue")
ax1.tick_params(axis="y", labelcolor="steelblue")

ax2 = ax1.twinx()
ax2.plot(xs, cumulative * 100, color="darkorange", marker="o",
         markersize=3, linewidth=1.8)
ax2.axhline(90, color="darkorange", linestyle=":", linewidth=1,
            label="90 % threshold")
ax2.set_ylabel("Cumulative Explained Variance (%)", color="darkorange")
ax2.tick_params(axis="y", labelcolor="darkorange")
ax2.set_ylim(0, 105)
ax2.legend(loc="center right")

# Annotate the 90%-threshold crossing
crossing = np.searchsorted(cumulative, 0.90)
if crossing < k:
    ax1.axvline(crossing + 1, color="gray", linestyle="--", linewidth=1)
    ax1.text(crossing + 1.3, ax1.get_ylim()[1] * 0.85,
             f"PC {crossing+1}\n→90%", fontsize=8, color="gray")

ax1.set_title(f"Scree plot  –  subject {SUBJECT}")
fig.tight_layout()
plt.show()

## 8 · PC1 vs PC2 scatter

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(
    scores[:, 0], scores[:, 1],
    c=np.arange(N), cmap="viridis",
    s=5, alpha=0.5, linewidths=0,
)
cb = plt.colorbar(sc, ax=ax)
cb.set_label("Gaussian index (proxy for body region)")
ax.set_xlabel(f"PC 1  ({ev_ratio[0]*100:.1f}% var)")
ax.set_ylabel(f"PC 2  ({ev_ratio[1]*100:.1f}% var)")
ax.set_title(f"PC1 vs PC2  –  subject {SUBJECT}  (N={N})")
fig.tight_layout()
plt.show()

## 9 · 3-D scatter (PC1, PC2, PC3)

In [ ]:
if k >= 3:
    fig = plt.figure(figsize=(8, 7))
    ax = fig.add_subplot(111, projection="3d")
    sc3 = ax.scatter(
        scores[:, 0], scores[:, 1], scores[:, 2],
        c=np.arange(N), cmap="plasma",
        s=4, alpha=0.4, linewidths=0,
    )
    plt.colorbar(sc3, ax=ax, pad=0.1, label="Gaussian index")
    ax.set_xlabel(f"PC1 ({ev_ratio[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({ev_ratio[1]*100:.1f}%)")
    ax.set_zlabel(f"PC3 ({ev_ratio[2]*100:.1f}%)")
    ax.set_title(f"3-D PCA scatter  –  subject {SUBJECT}")
    fig.tight_layout()
    plt.show()
else:
    print("Fewer than 3 PCs – skipping 3-D scatter.")

## 10 · Pairwise scatter matrix (PC 1–4)

In [ ]:
n_pair = min(k, 4)
fig, axes = plt.subplots(n_pair, n_pair, figsize=(10, 10))
colours = plt.cm.viridis(np.linspace(0, 1, N))

for row in range(n_pair):
    for col in range(n_pair):
        ax = axes[row, col]
        if row == col:
            # Diagonal: histogram of that PC
            ax.hist(scores[:, row], bins=40, color="steelblue", alpha=0.8)
            ax.set_title(f"PC{row+1}", fontsize=9)
        else:
            ax.scatter(scores[:, col], scores[:, row],
                       c=np.arange(N), cmap="viridis",
                       s=2, alpha=0.4, linewidths=0)
        if col == 0:
            ax.set_ylabel(f"PC{row+1}", fontsize=8)
        if row == n_pair - 1:
            ax.set_xlabel(f"PC{col+1}", fontsize=8)
        ax.tick_params(labelsize=6)

fig.suptitle(f"Pairwise PC scatter  –  subject {SUBJECT}", fontsize=12)
fig.tight_layout()
plt.show()

## 11 · PC loadings heatmap (top channels per PC)

In [ ]:
N_PC_SHOW  = min(k, 10)   # PCs to display as rows
TOP_CH     = 32           # top channels to show per PC (by absolute loading)

# Collect the union of top channels across the selected PCs
top_ch_set = set()
for pc_i in range(N_PC_SHOW):
    top_ch = np.argsort(np.abs(components[pc_i]))[-TOP_CH:]
    top_ch_set.update(top_ch.tolist())
top_ch_idx = sorted(top_ch_set)

loading_sub = components[:N_PC_SHOW][:, top_ch_idx]  # (N_PC_SHOW, len(top_ch_idx))

fig, ax = plt.subplots(figsize=(min(14, len(top_ch_idx) * 0.35 + 2), N_PC_SHOW * 0.55 + 1.5))
im = ax.imshow(loading_sub, aspect="auto", cmap="RdBu_r",
               vmin=-np.abs(loading_sub).max(), vmax=np.abs(loading_sub).max())
plt.colorbar(im, ax=ax, label="Loading weight")
ax.set_yticks(range(N_PC_SHOW))
ax.set_yticklabels([f"PC{i+1}" for i in range(N_PC_SHOW)], fontsize=8)
ax.set_xticks(range(len(top_ch_idx)))
ax.set_xticklabels(top_ch_idx, fontsize=6, rotation=90)
ax.set_xlabel("Channel index")
ax.set_title(f"PC loadings heatmap  –  subject {SUBJECT}\n"
             f"(top-{TOP_CH} channels per PC, first {N_PC_SHOW} PCs)")
fig.tight_layout()
plt.show()

## 12 · Save all figures
Run this cell to write every figure above to disk.

In [ ]:
OUT_DIR = FEAT_PATH.parent
OUT_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name):
    p = OUT_DIR / f"{SUBJECT}_{name}.png"
    fig.savefig(p, dpi=150, bbox_inches="tight")
    print(f"  Saved → {p}")

# Re-generate and save each figure
# --- channel stats ---
fig_ch, axes_ch = plt.subplots(2, 1, figsize=(14, 7))
axes_ch[0].bar(ch_idx, ch_means, yerr=ch_stds, color="teal", alpha=0.7, ecolor="gray", capsize=0, width=1.0)
axes_ch[0].axhline(0, color="black", linewidth=0.6)
axes_ch[0].set_title(f"Per-channel mean ± std  –  subject {SUBJECT}")
axes_ch[0].set_xlabel(f"Channel index (first {show} of {C})")
axes_ch[0].set_ylabel("Mean ± std")
axes_ch[1].bar(ch_idx, ch_stds, color="coral", alpha=0.8, width=1.0)
axes_ch[1].set_title("Per-channel standard deviation")
axes_ch[1].set_xlabel(f"Channel index (first {show} of {C})")
axes_ch[1].set_ylabel("Std")
fig_ch.tight_layout()
save_fig(fig_ch, "channel_stats")
plt.close(fig_ch)

# --- norm histogram ---
fig_norm, ax_norm = plt.subplots(figsize=(8, 4))
ax_norm.hist(norms, bins=60, color="mediumpurple", alpha=0.8, edgecolor="white", linewidth=0.3)
ax_norm.axvline(norms.mean(), color="red", linewidth=1.5, linestyle="--", label=f"mean={norms.mean():.3f}")
ax_norm.axvline(np.median(norms), color="orange", linewidth=1.5, linestyle="--", label=f"median={np.median(norms):.3f}")
ax_norm.set_title(f"Feature-vector norm distribution  –  subject {SUBJECT}")
ax_norm.legend()
fig_norm.tight_layout()
save_fig(fig_norm, "norm_hist")
plt.close(fig_norm)

# --- scree ---
fig_scree, ax_s1 = plt.subplots(figsize=(11, 4))
ax_s1.bar(np.arange(1, k+1), ev_ratio*100, color="steelblue", alpha=0.75, width=0.8)
ax_s1.set_xlabel("Principal Component"); ax_s1.set_ylabel("Explained Variance (%)", color="steelblue")
ax_s2 = ax_s1.twinx()
ax_s2.plot(np.arange(1, k+1), cumulative*100, color="darkorange", marker="o", markersize=3, linewidth=1.8)
ax_s2.set_ylabel("Cumulative (%)", color="darkorange"); ax_s2.set_ylim(0, 105)
ax_s1.set_title(f"Scree plot  –  subject {SUBJECT}")
fig_scree.tight_layout()
save_fig(fig_scree, "pca_scree")
plt.close(fig_scree)

# --- scatter 2D ---
fig_2d, ax_2d = plt.subplots(figsize=(7, 6))
sc2 = ax_2d.scatter(scores[:,0], scores[:,1], c=np.arange(N), cmap="viridis", s=5, alpha=0.5, linewidths=0)
plt.colorbar(sc2, ax=ax_2d, label="Gaussian index")
ax_2d.set_xlabel(f"PC1 ({ev_ratio[0]*100:.1f}%)"); ax_2d.set_ylabel(f"PC2 ({ev_ratio[1]*100:.1f}%)")
ax_2d.set_title(f"PC1 vs PC2  –  subject {SUBJECT}")
fig_2d.tight_layout()
save_fig(fig_2d, "pca_scatter2d")
plt.close(fig_2d)

# --- scatter 3D ---
if k >= 3:
    fig_3d = plt.figure(figsize=(8, 7))
    ax_3d = fig_3d.add_subplot(111, projection="3d")
    sc3b = ax_3d.scatter(scores[:,0], scores[:,1], scores[:,2], c=np.arange(N), cmap="plasma", s=4, alpha=0.4, linewidths=0)
    plt.colorbar(sc3b, ax=ax_3d, pad=0.1, label="Gaussian index")
    ax_3d.set_xlabel(f"PC1"); ax_3d.set_ylabel(f"PC2"); ax_3d.set_zlabel(f"PC3")
    ax_3d.set_title(f"3-D PCA scatter  –  subject {SUBJECT}")
    fig_3d.tight_layout()
    save_fig(fig_3d, "pca_scatter3d")
    plt.close(fig_3d)

# --- loadings heatmap ---
fig_hm, ax_hm = plt.subplots(figsize=(min(14, len(top_ch_idx)*0.35+2), N_PC_SHOW*0.55+1.5))
im2 = ax_hm.imshow(loading_sub, aspect="auto", cmap="RdBu_r", vmin=-np.abs(loading_sub).max(), vmax=np.abs(loading_sub).max())
plt.colorbar(im2, ax=ax_hm, label="Loading weight")
ax_hm.set_yticks(range(N_PC_SHOW)); ax_hm.set_yticklabels([f"PC{i+1}" for i in range(N_PC_SHOW)], fontsize=8)
ax_hm.set_xticks(range(len(top_ch_idx))); ax_hm.set_xticklabels(top_ch_idx, fontsize=6, rotation=90)
ax_hm.set_title(f"PC loadings heatmap  –  subject {SUBJECT}")
fig_hm.tight_layout()
save_fig(fig_hm, "pca_loadings")
plt.close(fig_hm)

print("\nAll figures saved.")